In [34]:
import pandas as pd

In [35]:
df = pd.read_csv('Xyzdata.csv')

In [36]:
df.drop(columns=['Unnamed: 0','Economic  Factor'],inplace=True)

In [37]:
df

,Locality,Average Price,Risk,min_price,max_price,Year,Quarter Period,Average Property Age (in years),Vacancy Rate (%),Occupancy Rate (%),Demand-Supply Ratio,Standard Deviation (?)
0,100 Feet Ring Road,10139,Moderate,7264,13013,2024,Jul-Sep,15,12.0,88.0,1.2,5.2
1,100 Feet Ring Road,10100,Moderate,7174,13026,2024,Apr-Jun,15,12.0,88.0,1.2,5.2
2,100 Feet Ring Road,10050,Moderate,7100,13000,2024,Jan-Mar,15,12.0,88.0,1.2,5.2
3,100 Feet Ring Road,10005,Moderate,7050,12950,2023,Oct-Dec,15,12.0,88.0,1.2,5.2
4,100 Feet Ring Road,9970,Moderate,7000,12900,2023,Jul-Sep,15,12.0,88.0,1.2,5.2
...,...,...,...,...,...,...,...,...,...,...,...,...
3932,Horamavu Agara,4919,Moderate,3508,6329,2019,Jul-Sep,10-15,18.0,82.0,1:1,0.6
3933,Horamavu Agara,4912,Moderate,3502,6322,2019,Apr-Jun,10-15,18.0,82.0,1:1,0.6
3934,Horamavu Agara,4459,Moderate,3430,5487,2019,Jan-Mar,10-15,18.0,82.0,1:1,0.6
3935,Horamavu Agara,4467,Moderate,3436,5498,2018,Oct-Dec,10-15,18.0,82.0,1:1,0.6


In [38]:
df2 = pd.read_csv("Economic FactorS Sentiment Analysis.csv")

In [39]:
df3 = pd.merge(df,df2,on='Locality')

In [40]:
df3.drop(columns=['Unnamed: 0','Sentiment Score','Economic  Factor'],inplace=True)

In [41]:
df3.to_csv('Trial Bangalore Data.csv')

In [42]:
#locality encoding and scaling 
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
encoder = LabelEncoder()
df3['Locality'] = encoder.fit_transform(df['Locality'])
scaler = MinMaxScaler(feature_range=(1,10))
df3[['Locality']] = scaler.fit_transform(df3[['Locality']])
joblib.dump(encoder,'Bangalore_re_locality_encoding.pkl')
joblib.dump(scaler,'Bangalore_re_locality_scaling.pkl')

['Bangalore_re_locality_scaling.pkl']

In [43]:
#prices scaling avg, min and max 
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
columns_to_scale = ['Average Price','min_price','max_price']
scaler = MinMaxScaler(feature_range=(1,10))
df3[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])
joblib.dump(scaler,'Bangalore_re_prices_scaling.pkl')

['Bangalore_re_prices_scaling.pkl']

In [44]:
#risk mapping
risk_mapping = {
    "High" : 7.5,
    "Low"   : 2.5,
    "Moderate" : 5.0
}

df3['Risk_mapped'] = df3['Risk'].map(risk_mapping)

In [45]:
#year - cyclic encoding 
import numpy as np
df3['Year_sin'] = np.sin(2 * np.pi * (df3['Year'] - df3['Year'].min()) / (df3['Year'].max() - df3['Year'].min()))
df3['Year_cos'] = np.cos(2 * np.pi * (df3['Year'] - df3['Year'].min()) / (df3['Year'].max() - df3['Year'].min()))

In [46]:
#quarter mapping
quarter_mapping = {
    "Jan-Mar" : 1,
    "Apr-Jun" : 2,
    "Jul-Sep" : 3,
    "Oct-Dec" : 4
}

df3['Quarter_mapped'] = df3['Quarter Period'].map(quarter_mapping)

In [47]:
#property age treatment as it is in range
def convert_age(age):
    if isinstance(age, str) and "-" in age:
        low, high = map(int, age.split("-"))
        return (low+high)/2
    return float(age)
df3['Average Property Age (in years)'] = df3['Average Property Age (in years)'].apply(convert_age)
print(df3["Average Property Age (in years)"].dtype)  
print(df3.head())

float64
   Locality  Average Price      Risk  min_price  max_price  Year  \
0       1.0       4.486076  Moderate   4.173374   4.658695  2024   
1       1.0       4.467169  Moderate   4.116975   4.663857  2024   
2       1.0       4.442930  Moderate   4.070603   4.653532  2024   
3       1.0       4.421115  Moderate   4.039270   4.633676  2023   
4       1.0       4.404148  Moderate   4.007938   4.613820  2023   

  Quarter Period  Average Property Age (in years)  Vacancy Rate (%)  \
0        Jul-Sep                             15.0              12.0   
1        Apr-Jun                             15.0              12.0   
2        Jan-Mar                             15.0              12.0   
3        Oct-Dec                             15.0              12.0   
4        Jul-Sep                             15.0              12.0   

   Occupancy Rate (%) Demand-Supply Ratio  Standard Deviation (?) Sentiment  \
0                88.0                 1.2                     5.2  Negative  

In [48]:
#vacny and occpuancy converion encoding
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
encoder = LabelEncoder()
columns_to_scale = ['Vacancy Rate (%)','Occupancy Rate (%)']
scaler = MinMaxScaler(feature_range=(1,10))
df3[columns_to_scale]=scaler.fit_transform(df3[columns_to_scale])
joblib.dump(scaler,'Bangalore_vacancy_and_occupancy_scaling.pkl')

['Bangalore_vacancy_and_occupancy_scaling.pkl']

In [49]:
#average property age in 1-10 range
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
encoder = LabelEncoder()
columns_to_scale = ['Average Property Age (in years)']
scaler = MinMaxScaler(feature_range=(1,10))
df3[columns_to_scale]=scaler.fit_transform(df3[columns_to_scale])
joblib.dump(scaler,'Bangalore_propertyage_scaling.pkl')

['Bangalore_propertyage_scaling.pkl']

In [50]:
#sentiment treatment
sentiment_mapping = {
    "Negative": 2.5,
    "Positive": 5,
    "Neutral": 7.5,
}
df3['sentiment_Mapped'] = df3['Sentiment'].map(sentiment_mapping)

In [51]:
#quarter mapping
quarter_mapping = {
    "Jan-Mar" : 1,
    "Apr-Jun" : 2,
    "Jul-Sep" : 3,
    "Oct-Dec" : 4
}

df3['Quarter_mapped'] = df3['Quarter Period'].map(quarter_mapping)

In [52]:
df3.drop(columns=['Risk','Year','Quarter Period','Sentiment'],inplace=True)

In [53]:
df3.dropna()

,Locality,Average Price,min_price,max_price,Average Property Age (in years),Vacancy Rate (%),Occupancy Rate (%),Demand-Supply Ratio,Standard Deviation (?),Risk_mapped,Year_sin,Year_cos,Quarter_mapped,sentiment_Mapped
0,1.000000,4.486076,4.173374,4.658695,7.428571,3.1,7.9,1.2,5.2,5.0,-2.449294e-16,1.0,3,2.5
1,1.000000,4.467169,4.116975,4.663857,7.428571,3.1,7.9,1.2,5.2,5.0,-2.449294e-16,1.0,2,2.5
2,1.000000,4.442930,4.070603,4.653532,7.428571,3.1,7.9,1.2,5.2,5.0,-2.449294e-16,1.0,1,2.5
3,1.000000,4.421115,4.039270,4.633676,7.428571,3.1,7.9,1.2,5.2,5.0,-8.660254e-01,0.5,4,2.5
4,1.000000,4.404148,4.007938,4.613820,7.428571,3.1,7.9,1.2,5.2,5.0,-8.660254e-01,0.5,3,2.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3932,4.559322,1.955508,1.819663,2.004324,6.142857,4.9,6.1,1:1,0.6,5.0,8.660254e-01,0.5,3,7.5
3933,4.559322,1.952114,1.815903,2.001544,6.142857,4.9,6.1,1:1,0.6,5.0,8.660254e-01,0.5,2,7.5
3934,4.559322,1.732507,1.770784,1.669947,6.142857,4.9,6.1,1:1,0.6,5.0,8.660254e-01,0.5,1,7.5
3935,4.559322,1.736386,1.774544,1.674315,6.142857,4.9,6.1,1:1,0.6,5.0,0.000000e+00,1.0,4,7.5


In [54]:
df3.to_csv('data_prepreocessded.csv')

In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

features = [
    'Locality', 'min_price', 'max_price', 'Average Property Age (in years)', 
    'Vacancy Rate (%)', 'Occupancy Rate (%)', 
    'Standard Deviation (?)', 'Risk_mapped', 'Year_sin', 'Year_cos', 
    'Quarter_mapped', 'sentiment_Mapped'
]

target = 'Average Price'

X = df3[features]
y = df3[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MSE: {mse}")
print(f"R² Score: {r2}")

joblib.dump(model, 'Bangalore_price_prediction_model.pkl')
print("Model saved as 'Bangalore_price_prediction_model.pkl'")

MSE: 0.014336907232343253
R² Score: 0.9933204135146112
Model saved as 'Bangalore_price_prediction_model.pkl'


In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

features = [
    'Locality', 'min_price', 'max_price', 'Average Property Age (in years)', 
    'Vacancy Rate (%)', 'Occupancy Rate (%)', 
    'Standard Deviation (?)', 'Risk_mapped', 'Year_sin', 'Year_cos', 
    'Quarter_mapped', 'sentiment_Mapped'
]

target = 'Average Price'

X = df3[features]
y = df3[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MSE: {mse}")
print(f"R² Score: {r2}")

joblib.dump(model, 'Bangalore_re_price_prediction_model.pkl')
print("Model saved as 'Bangalore_price_prediction_model.pkl'")

MSE: 0.014336907232343253
R² Score: 0.9933204135146112
Model saved as 'Bangalore_price_prediction_model.pkl'


In [57]:
y_train_pred = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
print(f"Training Mean Squared Error: {train_mse}")
print(f"Training R² Score: {train_r2}")

Training Mean Squared Error: 0.0014623361419288613
Training R² Score: 0.999268541752458


In [58]:
print("Feature Importances:")
for feature, importance in zip(features, model.feature_importances_):
    print(f"{feature}: {importance:.4f}")

Feature Importances:
Locality: 0.0009
min_price: 0.0253
max_price: 0.9711
Average Property Age (in years): 0.0002
Vacancy Rate (%): 0.0001
Occupancy Rate (%): 0.0001
Standard Deviation (?): 0.0007
Risk_mapped: 0.0001
Year_sin: 0.0007
Year_cos: 0.0004
Quarter_mapped: 0.0002
sentiment_Mapped: 0.0001


In [59]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f"Cross-Validation R² Scores: {cv_scores}")
print(f"Average CV R²: {cv_scores.mean():.4f}, Std: {cv_scores.std():.4f}")

Cross-Validation R² Scores: [0.97140949 0.98703002 0.99752208 0.99602626 0.99566893]
Average CV R²: 0.9895, Std: 0.0098
